# Занятие 3. Цвет, гистограммы, пороги, морфология

**Курс «Введение в компьютерное зрение» · Innopolis University · Fall 2026**
Лекция-опора: **L3 «Цвет, гистограммы и точечные операции»** · **90 минут в классе** · ведёт ассистент

---

Сегодня собираем **цветовой трекер**, который переживает смену освещения: маска по цвету
в HSV → морфология → связные компоненты → рамка (bbox) — четыре вызова на кадр, и они работают
на видео. Потом меряем **четыре способа бинаризации** страницы с неровным светом, подбираем
параметры адаптивного порога **по метрике** и находим, где подобранное ломается.

Заготовка начинается ровно на демо 2.1 и 2.2 лекции: вы это уже видели, теперь — руками.

| Минуты | Что происходит |
|--------|----------------|
| 0–15 | **Quiz 0** в Moodle (L1–L2, не оценивается) |
| 15–25 | Ассистент разбирает опорный пример: границы с эталона, RGB против HSV, конвейер маска → bbox |
| 25–65 | Вы делаете **TODO 1–3** |
| 65–75 | Разбор решения |
| 75–85 | **Мост к ДЗ 1 «Фотолаборатория»** (выдаётся на этой неделе) |
| 85–90 | Зачёт |

Ничего сдавать не нужно: **зачёт ставится в классе** по факту работы (2 % итоговой оценки).
⭐ — необязательная звёздочка. Решение публикуется сразу после занятия.

## 0. Проверка окружения

Если ячейка ругается — зовите ассистента сразу, не тратьте время занятия.

In [ ]:
REPO_URL = "https://github.com/afanasyspb/iu-intro-cv.git"     # адрес репозитория курса (для Colab)

import sys, subprocess
try:
    import cvcourse
except ImportError:
    if "google.colab" in sys.modules:      # Colab: пакет курса ставится один раз за сессию
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "opencv-contrib-python==4.14.0.94", "git+" + REPO_URL], check=True)
        import cvcourse
    else:
        raise ImportError("пакет курса не установлен: из корня репозитория выполните "
                          "pip install -r requirements.txt   (docs/setup-guide.md)")

import os, time
import cv2, numpy as np
import matplotlib.pyplot as plt
from cvcourse import io as cio, viz, metrics

print("OpenCV", cv2.__version__, "| NumPy", np.__version__, "| Colab:", cvcourse.IN_COLAB)
assert cv2.__version__.startswith("4.14"), "курс собран на OpenCV 4.14.0.94"

## 1. Берём кадры

Два кадра из лекции: фото здания ИУ с **пурпурным мячом** при дневном свете (`ball-day.jpg`) и тот же
кадр **вдвое тусклее и чуть теплее** (`ball-lamp.jpg`). Мяч в кадре нарисован, поэтому у нас есть
**эталон**: круг с центром (190, 560) и радиусом 58 px — по нему будем считать, какую долю мяча
нашли (recall) и какая доля найденного — мяч (precision).

Источники по порядку: репозиторий рядом → скачать с GitHub (Colab) → синтетический фон
(работает всегда, числа другие).

In [ ]:
LECTURE_DIR = "lectures/L03-color-histograms/demo"
CANDIDATE_DIRS = [
    "../../" + LECTURE_DIR,                                # публичный репозиторий: labs/lab03-color/
    "../../../iu-intro-cv/" + LECTURE_DIR,                 # ноутбук открыт из iu-intro-cv-ta
    "data",                                                # уже скачанное
]
RAW_URL = "https://raw.githubusercontent.com/afanasyspb/iu-intro-cv/main/" + LECTURE_DIR + "/"
BALL = (190, 560, 58)                                      # эталон: центр (x, y) и радиус мяча
H, W = 640, 960


def paint_ball(img, cx, cy, r, gain=(1.0, 1.0, 1.0)):
    """Рисует пурпурный мяч с тенью и бликом; gain — множители каналов B, G, R (освещение)."""
    yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
    nx, ny = (xx - cx) / float(r), (yy - cy) / float(r)
    inside = nx ** 2 + ny ** 2 <= 1.0
    nz = np.sqrt(np.clip(1 - nx ** 2 - ny ** 2, 0, 1))
    L = np.array([-0.45, -0.60, 0.66]); L /= np.linalg.norm(L)
    shade = 0.45 + 0.55 * np.clip(nx * L[0] + ny * L[1] + nz * L[2], 0, 1)
    spec = 0.55 * np.exp(-((nx + 0.35) ** 2 + (ny + 0.45) ** 2) / 0.02)
    ball = (np.array([200, 40, 235], np.float32)[None, None, :] * shade[..., None] + 255 * spec[..., None])
    out = img.astype(np.float32).copy()
    out[inside] = (ball * np.array(gain, np.float32))[inside]
    return out, inside


def make_scene(seed=0):
    """Запасной фон вместо фото: прямоугольники без пурпурного."""
    rng = np.random.default_rng(seed)
    img = np.full((H, W, 3), 170, np.uint8)
    for _ in range(24):
        c = tuple(int(v) for v in rng.integers(30, 230, 3)); c = (c[0], max(c[1], 90), c[2])   # G ≥ 90: не пурпур
        x, y = int(rng.integers(0, W - 200)), int(rng.integers(0, H - 200))
        cv2.rectangle(img, (x, y), (x + int(rng.integers(60, 200)), y + int(rng.integers(60, 200))), c, -1)
    return img


def load_frames():
    for d in CANDIDATE_DIRS:
        if all(os.path.exists(os.path.join(d, f)) for f in ("ball-day.jpg", "ball-lamp.jpg")):
            return cio.imread(os.path.join(d, "ball-day.jpg")), cio.imread(os.path.join(d, "ball-lamp.jpg")), d
    try:                                                   # Colab: репозитория рядом нет — качаем
        import urllib.request
        os.makedirs("data", exist_ok=True)
        for f in ("ball-day.jpg", "ball-lamp.jpg"):
            urllib.request.urlretrieve(RAW_URL + f, os.path.join("data", f))
        return cio.imread("data/ball-day.jpg"), cio.imread("data/ball-lamp.jpg"), RAW_URL
    except Exception as e:
        print("кадры не скачались (%s) — синтетический фон" % type(e).__name__)
        rng = np.random.default_rng(3)
        day, _ = paint_ball(make_scene(), *BALL)
        day = np.clip(day + rng.normal(0, 2.5, day.shape), 0, 255).astype(np.uint8)
        lamp = np.clip(day * np.array([0.42, 0.45, 0.50], np.float32) + rng.normal(0, 2.5, day.shape), 0, 255).astype(np.uint8)
        return day, lamp, "синтетика"


day, lamp, SRC = load_frames()
assert day.shape == (H, W, 3) == lamp.shape, "ожидались кадры 960 × 640"
yy, xx = np.mgrid[:H, :W]
gt_ball = (xx - BALL[0]) ** 2 + (yy - BALL[1]) ** 2 <= BALL[2] ** 2          # эталонная маска мяча (bool)
gt_box = (BALL[0] - BALL[2], BALL[1] - BALL[2], 2 * BALL[2] + 1, 2 * BALL[2] + 1)
print("источник:", SRC, "· кадры", day.shape, "· эталон: %d px мяча" % gt_ball.sum())
viz.grid({"«день»": day, "«лампа»": lamp, "эталон мяча": gt_ball.astype(np.uint8) * 255}, cols=3, size=3.6)

## 2. Опорный пример — разбирает ассистент

Это демо 2.2 лекции и слайд 40. Здесь всё написано, задача — **понять каждую строку**.

**Откуда берутся границы `inRange`.** С эталонного кадра «день»: берём пиксели мяча, по каждому
каналу — 1-й и 99-й процентили ± 10 кодов запаса. Так строим параллелепипед и в **RGB**, и в **HSV** —
и применяем оба к кадру «лампа», где свет другой. Две метрики против эталона:
**recall** — какую долю мяча нашли, **precision** — какая доля найденного действительно мяч.

In [ ]:
def box_from(pixels, margin=10, p=(1, 99)):
    """Границы (lo, hi) по каналам: процентили p пикселей эталона ± margin, обрезано в 0…255."""
    lo = np.clip(np.floor(np.percentile(pixels, p[0], axis=0) - margin), 0, 255)
    hi = np.clip(np.ceil(np.percentile(pixels, p[1], axis=0) + margin), 0, 255)
    return tuple(int(v) for v in lo), tuple(int(v) for v in hi)


def hsv_box_from(bgr_pixels, margin=10, p=(1, 99)):
    """То же в HSV: H и S — с эталона, V — только отсечь чёрное (V ≥ 40): яркость меняет свет."""
    hsv = cv2.cvtColor(bgr_pixels.reshape(-1, 1, 3), cv2.COLOR_BGR2HSV).reshape(-1, 3)
    lo, hi = box_from(hsv, margin, p)
    return (lo[0], lo[1], 40), (min(hi[0], 179), hi[1], 255)


def recall_precision(mask, gt):
    """Доля найденного эталона и доля эталона среди найденного, в процентах."""
    m, tp = mask > 0, ((mask > 0) & gt).sum()
    return 100.0 * tp / max(gt.sum(), 1), 100.0 * tp / max(m.sum(), 1)


rgb_lo, rgb_hi = box_from(day[gt_ball])                    # границы — с кадра «день»
hsv_lo, hsv_hi = hsv_box_from(day[gt_ball])
print("RGB (B, G, R):", rgb_lo, "…", rgb_hi)
print("HSV          :", hsv_lo, "…", hsv_hi, "   ← H и S с эталона, V открыт")

print("\n%-22s %10s %10s" % ("маска на «лампе»", "recall", "precision"))
masks = {"RGB-параллелепипед": cv2.inRange(lamp, np.array(rgb_lo, np.uint8), np.array(rgb_hi, np.uint8)),
         "HSV, V ≥ 40":        cv2.inRange(cv2.cvtColor(lamp, cv2.COLOR_BGR2HSV),
                                           np.array(hsv_lo, np.uint8), np.array(hsv_hi, np.uint8))}
for name, m in masks.items():
    print("%-22s %8.1f %% %8.1f %%" % ((name,) + recall_precision(m, gt_ball)))
viz.grid({"«лампа»": lamp, "RGB с кадра «день»": masks["RGB-параллелепипед"], "HSV с кадра «день»": masks["HSV, V ≥ 40"]},
         cols=3, size=3.6)

**Конвейер трекера** — четыре вызова (слайд 45): `inRange` → открытие 5 × 5 (крапинки прочь) →
закрытие 9 × 9 (дыру от блика зашить) → `connectedComponentsWithStats` → крупнейшая компонента → bbox.
Мерим рамку по **IoU** (пересечение на объединение) с эталонной.

In [ ]:
k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
k9 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))

raw = masks["HSV, V ≥ 40"]
clean = cv2.morphologyEx(raw, cv2.MORPH_OPEN, k5)                     # крапинки прочь
clean = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, k9)                  # дыры зашить
n, labels, stats, cent = cv2.connectedComponentsWithStats(clean)      # метка 0 — фон
i = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))                   # крупнейшая компонента
x, y, w, h, area = (int(v) for v in stats[i])

holes = lambda m: cv2.connectedComponents(255 - m)[0] - 2             # компоненты фона минус внешний
print("сырая маска: компонент %d, дыр %d · после морфологии: компонент %d, дыр %d"
      % (cv2.connectedComponentsWithStats(raw)[0] - 1, holes(raw), n - 1, holes(clean)))
print("bbox (%d, %d, %d × %d), площадь %d px, центроид (%.1f, %.1f), IoU с эталоном %.2f"
      % (x, y, w, h, area, cent[i][0], cent[i][1], metrics.iou((x, y, w, h), gt_box)))

shown = lamp.copy()
cv2.rectangle(shown, (x, y), (x + w, y + h), viz.ORANGE, 2)
crop = (slice(y - 40, y + h + 40), slice(x - 40, x + w + 40))
viz.grid({"сырая маска": raw[crop], "после открытия и закрытия": clean[crop], "bbox на кадре": shown[crop]}, cols=3, size=3.4)

### Гистограммы: какой канал сдвинул свет

`cv2.calcHist` умеет считать гистограмму **только под маской** — посчитаем H, S, V мяча на обоих
кадрах. Это ответ на вопрос «почему V не берём с эталона».

In [ ]:
mask8 = gt_ball.astype(np.uint8)
fig, axes = plt.subplots(1, 3, figsize=(11, 2.8))
for k, (name, top) in enumerate((("H", 180), ("S", 256), ("V", 256))):
    for frame, color, label in ((day, "#0070C0", "день"), (lamp, "#C55A11", "лампа")):
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [k], mask8, [top], [0, top]).ravel()      # гистограмма под маской мяча
        axes[k].plot(hist, color=color, label=label)
        lo, hi = np.percentile(hsv[..., k][gt_ball], [1, 99])
        print("%s %-6s: 1–99 %% в %3d…%3d" % (name, label, lo, hi))
    axes[k].set_title("канал " + name, fontsize=10); axes[k].grid(alpha=.3); axes[k].legend(fontsize=8)
fig.tight_layout(); plt.show()

### Вопрос, на который отвечаем вслух

Границы сняты с одного кадра и применены к другому. В RGB найдено 39 % мяча, и лишь 2 % найденного —
мяч; в HSV — 97 % и 100 %. По гистограммам видно, **какой** канал сдвинул свет и **какие** остались
на месте. Что было бы, если бы мы взяли с эталона и диапазон V?

---

## TODO 1 — маска по цвету *(≈ 10 минут)*

Напишите `color_mask(bgr, lo, hi, open_k=5, close_k=9)`: кадр BGR → HSV → `cv2.inRange(lo, hi)` →
открытие эллипсом `open_k × open_k` → закрытие `close_k × close_k`; результат — маска `uint8` 0/255.
`open_k = 0` или `close_k = 0` — соответствующий шаг пропускается.

**Красный.** В OpenCV hue лежит в 0…179, и красный **рвётся через ноль**: одна маска `[0, 10]` теряет
38 % пикселей (слайд 54). Договоримся: если `lo[0] > hi[0]` — диапазон идёт **через 179**: две маски
`[lo[0], 179]` и `[0, hi[0]]` и `cv2.bitwise_or`.

> **Подсказка.** `cv2.inRange` ждёт границы массивами `np.uint8` (или кортежами целых).
> `cv2.morphologyEx(m, cv2.MORPH_OPEN, k)` — открытие; ядро — `cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))`.

In [ ]:
def color_mask(bgr, lo, hi, open_k=5, close_k=9):
    """HSV-маска по границам (lo, hi) + открытие/закрытие. lo[0] > hi[0] — диапазон H через 179. → uint8 0/255."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 7–10 строк: cvtColor в HSV; inRange (две маски и bitwise_or, если lo[0] > hi[0]); morphologyEx OPEN, CLOSE
    raise NotImplementedError("TODO 1")
    # ───────────────────────────────────────────────────────────


m_lamp = color_mask(lamp, hsv_lo, hsv_hi)
assert m_lamp.shape == (H, W) and m_lamp.dtype == np.uint8, "маска — uint8 формы (h, w)"
assert set(np.unique(m_lamp)) <= {0, 255}, "значения маски — только 0 и 255"
rec, prec = recall_precision(m_lamp, gt_ball)
assert rec > 90 and prec > 99, "на «лампе» должно найтись > 90 %% мяча при точности > 99 %% (сейчас %.1f / %.1f)" % (rec, prec)
assert cv2.connectedComponentsWithStats(m_lamp)[0] - 1 == 1, "после открытия и закрытия должна остаться ровно одна компонента"
assert np.array_equal(color_mask(lamp, hsv_lo, hsv_hi, 0, 0), raw), "с open_k = close_k = 0 должна получаться сырая inRange-маска"

rng = np.random.default_rng(0)                                       # красный диск на тёмном фоне + шум
red = np.full((120, 160, 3), 40, np.uint8); cv2.circle(red, (80, 60), 35, (30, 30, 220), -1)
red = np.clip(red + rng.normal(0, 8, red.shape), 0, 255).astype(np.uint8)
disc = np.zeros((120, 160), bool); disc[(np.mgrid[:120, :160][1] - 80) ** 2 + (np.mgrid[:120, :160][0] - 60) ** 2 <= 35 ** 2] = True
one = recall_precision(color_mask(red, (0, 100, 40), (10, 255, 255)), disc)[0]
two = recall_precision(color_mask(red, (170, 100, 40), (10, 255, 255)), disc)[0]
assert two > 95, "красный через 0/179: lo[0] > hi[0] должно давать объединение двух масок (найдено %.1f %%)" % two
print("TODO 1 ✔  · «лампа»: найдено %.1f %% мяча, точность %.1f %%, компонент 1" % (rec, prec))
print("           · красный диск: одна маска [0, 10] находит %.0f %%, через 0/179 — %.1f %%" % (one, two))

### Порядок операций — проверяем измерением

Лекция говорит: открытие, потом закрытие. Selfcheck-слайд 58 спрашивает, что изменится, если поменять
порядок. Не спорим — считаем на кадре «лампа» с теми же границами.

In [ ]:
print("%-20s %8s %10s %6s %8s" % ("вариант", "recall", "precision", "дыр", "площадь"))
for name, kw in (("сырая inRange", dict(open_k=0, close_k=0)), ("только открытие 5", dict(close_k=0)),
                 ("только закрытие 9", dict(open_k=0)), ("открытие → закрытие", {})):
    m = color_mask(lamp, hsv_lo, hsv_hi, **kw)
    print("%-20s %6.1f %% %8.1f %% %6d %8d" % ((name,) + recall_precision(m, gt_ball) + (holes(m), int((m > 0).sum()))))
m = cv2.morphologyEx(cv2.morphologyEx(raw, cv2.MORPH_CLOSE, k9), cv2.MORPH_OPEN, k5)
print("%-20s %6.1f %% %8.1f %% %6d %8d" % (("закрытие → открытие",) + recall_precision(m, gt_ball) + (holes(m), int((m > 0).sum()))))

### Вопрос, на который отвечаем вслух

Оба порядка убирают все дыры и оставляют одну компоненту, но «открытие → закрытие» съедает около 5 %
мяча по кромке, а обратный порядок — нет. Почему тогда лекция (и все туториалы) начинают с открытия?
Подсказка: представьте крапинку, **прилипшую** к объекту, и подумайте, что с ней сделает закрытие первым.

## TODO 2 — крупнейшая компонента и трекер *(≈ 15 минут)*

Напишите `largest_component(mask, min_area=200)`: по `cv2.connectedComponentsWithStats` найти
**крупнейшую** компоненту (метка 0 — фон, её пропускаем) и вернуть пару
`((x, y, w, h), (cx, cy))` — рамку и центроид. Если компонент нет или крупнейшая меньше `min_area`
пикселей — вернуть `None`: трекер должен уметь сказать «объекта в кадре нет», а не хвататься за мусор.

> **Подсказка.** `stats[:, cv2.CC_STAT_AREA]` — площади всех компонент, включая фон в строке 0;
> `1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])` — индекс крупнейшей **не-фоновой**.
> Строка `stats[i]` — это `x, y, w, h, area`; `centroids[i]` — `(cx, cy)` с дробями.

In [ ]:
def largest_component(mask, min_area=200):
    """Крупнейшая компонента маски: ((x, y, w, h), (cx, cy)); None, если её нет или она меньше min_area."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 6–8 строк: connectedComponentsWithStats; если n < 2 — None; argmax по площади без фона; проверка min_area
    raise NotImplementedError("TODO 2")
    # ───────────────────────────────────────────────────────────


box, (cx, cy) = largest_component(m_lamp)
err = float(np.hypot(cx - BALL[0], cy - BALL[1]))
assert metrics.iou(box, gt_box) > 0.9, "рамка крупнейшей компоненты должна совпадать с эталонной (IoU > 0.9)"
assert err < 3, "центроид должен лежать в 3 px от центра мяча (сейчас %.1f px)" % err
assert largest_component(np.zeros((50, 50), np.uint8)) is None, "пустая маска → None"
two_blobs = np.zeros((100, 100), np.uint8); two_blobs[10:20, 10:20] = 255; two_blobs[40:90, 40:90] = 255
assert largest_component(two_blobs)[0] == (40, 40, 50, 50), "из двух компонент нужна крупнейшая, а не первая"
assert largest_component(two_blobs[:30, :30]) is None, "компонента 10 × 10 меньше min_area = 200 → None"
assert largest_component(two_blobs[:30, :30], min_area=50)[0] == (10, 10, 10, 10), "порог min_area — параметр"
print("TODO 2 ✔  · bbox %s, центроид (%.1f, %.1f), IoU %.2f, ошибка центра %.1f px" % (box, cx, cy, metrics.iou(box, gt_box), err))

### Трекер на видео

Теперь те же два вызова — на каждом кадре. **Границы калибруем один раз** по первому кадру
(`hsv_box_from` — как в опорном примере) и больше не трогаем: свет по ходу видео меняется, а границы — нет.

Источник видео:

- **веб-камера** (локально): ячейка 2 секунды показывает квадрат калибровки — **заполните его
  цветным предметом** (маркер, чехол, обложка), потом 4 секунды водите предметом по кадру;
- **нет камеры** (Colab, `CVCOURSE_NO_CAMERA=1`): синтетическое видео — мяч летит по фото ИУ по дуге,
  уменьшается, а свет проходит путь «день → сумерки → день» (яркость до 0.45). Для синтетики
  известна **истинная траектория**, поэтому ошибку трекера можно измерить в пикселях.

In [ ]:
N_CALIB, N_TRACK = 30, 60                                  # кадров на калибровку и на трекинг (камера)
SOURCE = "synthetic"
frames, truth = [], None
if not os.environ.get("CVCOURSE_NO_CAMERA"):
    try:
        cam = list(cio.video_frames(0, max_frames=N_CALIB + N_TRACK, resize=(640, 360)))
        if len(cam) < N_CALIB + N_TRACK:
            raise RuntimeError("камера отдала только %d кадров" % len(cam))
        ch, cw = cam[0].shape[:2]
        calib_roi = (slice(ch // 2 - 40, ch // 2 + 40), slice(cw // 2 - 40, cw // 2 + 40))    # квадрат 80 × 80 в центре
        calib_pixels = cam[N_CALIB - 1][calib_roi].reshape(-1, 3)
        frames, SOURCE = cam[N_CALIB:], "camera"
    except Exception as e:
        print("камера недоступна (%s) — синтетическое видео" % type(e).__name__)

if SOURCE == "synthetic":
    rng = np.random.default_rng(5)
    bg = day.copy(); bg[490:, 120:300] = day[490:, 300:480]                        # фон «день» без мяча: мостовая справа
    truth = []
    for t in range(40):
        s = t / 39.0
        cx_t, cy_t, r_t = 120 + 720 * s, 330 + 200 * np.sin(2 * np.pi * s), 58 * (1 - 0.35 * s)
        g = 1 - 0.55 * np.sin(np.pi * s)                                       # свет: день → сумерки → день
        gain = (0.85 * g, g, g + 0.08 * (1 - g))                              # в сумерках чуть теплее
        f, _ = paint_ball(bg.astype(np.float32) * np.array(gain, np.float32), cx_t, cy_t, r_t, gain)
        frames.append(np.clip(f + rng.normal(0, 2.5, f.shape), 0, 255).astype(np.uint8))
        truth.append((cx_t, cy_t, r_t))
    calib_pixels = frames[0][(xx - truth[0][0]) ** 2 + (yy - truth[0][1]) ** 2 <= truth[0][2] ** 2]

lo_t, hi_t = hsv_box_from(calib_pixels)
print("источник: %s · %d кадров %s · границы HSV с первого кадра: %s … %s" % (SOURCE, len(frames), frames[0].shape[:2], lo_t, hi_t))
if hi_t[1] < 80 or hi_t[0] - lo_t[0] > 120:
    print("⚠ в квадрате калибровки не было цветного предмета: S ≤ %d, H почти весь круг — трекер будет ловить серое."
          " Повторите ячейку, заполнив квадрат предметом." % hi_t[1])
viz.grid({"кадр 0": frames[0], "середина": frames[len(frames) // 2], "последний": frames[-1]}, cols=3, size=3.6)

In [ ]:
def track(frames, mask_fn):
    """Прогоняет трекер по кадрам: список результатов largest_component (или None) на кадр."""
    return [largest_component(mask_fn(f)) for f in frames]


t0 = time.perf_counter()
hits = track(frames, lambda f: color_mask(f, lo_t, hi_t))
ms = 1000 * (time.perf_counter() - t0) / len(frames)
found = [h for h in hits if h is not None]
print("найден на %d из %d кадров · %.1f мс на кадр" % (len(found), len(frames), ms))

shown = frames[-1].copy()
pts = np.array([[c[0], c[1]] for _, c in found], np.int32)
if len(pts) > 1:
    cv2.polylines(shown, [pts], False, viz.BLUE, 2)                          # траектория центроидов
for i in (0, len(frames) // 2, len(frames) - 1):
    if hits[i]:
        (x, y, w, h), _ = hits[i]
        cv2.rectangle(shown, (x, y), (x + w, y + h), viz.ORANGE, 2)
viz.show(shown, "траектория и рамки на кадрах 0, %d, %d" % (len(frames) // 2, len(frames) - 1), size=7)

if SOURCE == "synthetic":
    ok = [i for i, h in enumerate(hits) if h and metrics.iou(h[0], (truth[i][0] - truth[i][2], truth[i][1] - truth[i][2], 2 * truth[i][2], 2 * truth[i][2])) > 0.5]
    errs = [np.hypot(hits[i][1][0] - truth[i][0], hits[i][1][1] - truth[i][1]) for i in ok]
    assert len(ok) == len(frames), "на синтетике мяч должен быть пойман (IoU > 0.5) на всех кадрах, поймано %d" % len(ok)
    assert np.mean(errs) < 3, "средняя ошибка центроида должна быть < 3 px (сейчас %.1f)" % np.mean(errs)
    print("IoU > 0.5 на %d из %d кадров · ошибка центроида: средняя %.2f px, максимум %.2f px" % (len(ok), len(frames), np.mean(errs), np.max(errs)))
else:
    assert len(found) >= 1, "объект не найден ни на одном кадре: проверьте, что предмет заполнял квадрат калибровки"
    print("свой предмет: чем меньше «дыр» в списке найденных кадров, тем лучше границы; сравните с синтетикой ниже")

### Три трекера — одно видео

Границы H и S с эталона, **V открыт** — это правило со слайда 39. Проверяем, чего оно стоит: тот же
трекер с границами V, тоже снятыми с первого кадра, и трекер в RGB. Считаем кадры, где рамка совпала
с истиной (IoU > 0.5), и ошибку центра. На камере истины нет — считаем только найденные кадры.

In [ ]:
hsv_c = cv2.cvtColor(calib_pixels.reshape(-1, 1, 3), cv2.COLOR_BGR2HSV).reshape(-1, 3)
lo_c, hi_c = box_from(hsv_c); hi_c = (min(hi_c[0], 179), hi_c[1], hi_c[2])       # V — тоже с эталона
rgb_lo_t, rgb_hi_t = box_from(calib_pixels)

def rgb_mask(f):
    m = cv2.inRange(f, np.array(rgb_lo_t, np.uint8), np.array(rgb_hi_t, np.uint8))
    return cv2.morphologyEx(cv2.morphologyEx(m, cv2.MORPH_OPEN, k5), cv2.MORPH_CLOSE, k9)

trackers = {"HSV, V ≥ 40 (правило)": lambda f: color_mask(f, lo_t, hi_t),
            "HSV, V с эталона":      lambda f: color_mask(f, lo_c, hi_c),
            "RGB-параллелепипед":     rgb_mask}
print("%-24s %10s %14s" % ("трекер", "найден", "IoU > 0.5 / ошибка"))
summary = {}
for name, fn in trackers.items():
    hs = track(frames, fn)
    n_found = sum(h is not None for h in hs)
    if SOURCE == "synthetic":
        ok = [i for i, h in enumerate(hs) if h and metrics.iou(h[0], (truth[i][0] - truth[i][2], truth[i][1] - truth[i][2], 2 * truth[i][2], 2 * truth[i][2])) > 0.5]
        e = np.mean([np.hypot(hs[i][1][0] - truth[i][0], hs[i][1][1] - truth[i][1]) for i in ok]) if ok else float("nan")
        summary[name] = (len(ok), e)
        print("%-24s %5d/%-4d %8d/%d  %5.1f px" % (name, n_found, len(frames), len(ok), len(frames), e))
    else:
        print("%-24s %5d/%-4d %14s" % (name, n_found, len(frames), "—"))
if SOURCE == "synthetic":
    assert summary["HSV, V ≥ 40 (правило)"][0] > summary["HSV, V с эталона"][0] > summary["RGB-параллелепипед"][0], \
        "ожидался порядок: V открыт > V с эталона > RGB"

### Вопрос, на который отвечаем вслух

RGB-трекер теряет мяч даже на **первом** кадре, при полном свете, — хотя границы сняты именно с него.
Посмотрите на precision RGB-маски в опорном примере и подумайте, **какую** компоненту выбирает
`largest_component`. А почему трекер с границами V «с эталона» теряет мяч именно в середине видео?

---

## TODO 3 — порог по метрике *(≈ 15 минут)*

Вторая задача лекции: страница текста, свет падает слева и к правому краю гаснет до 0.24.
Страницы **синтетические** — тот же генератор, что в демо 2.1, поэтому и числа те же; заодно у нас есть
эталон (`gt`: чернила 0, бумага 255) и метрика — **доля верных пикселей**. Четвёртая страница —
с крупным чёрным логотипом — понадобится через минуту.

In [ ]:
def make_pages():
    """Страница текста: эталон, ровный свет + шум, градиент света + шум, и та же под лампой с логотипом."""
    rng = np.random.default_rng(1)
    page = np.full((480, 640), 255, np.uint8)
    lines = ["Introduction to Computer Vision", "Innopolis University, Fall 2026",
             "Lecture 3: colour, histograms,", "thresholds and morphology.",
             "Otsu (1979): the threshold that", "maximises between-class variance.",
             "Adaptive: T(x, y) = local mean - C", "CLAHE: clip limit 2, tiles 8 x 8"]
    for i, t in enumerate(lines):
        cv2.putText(page, t, (26, 58 + 54 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.8, 0, 2, cv2.LINE_AA)
    gt = np.where(page < 128, 0, 255).astype(np.uint8)
    flat = np.clip(page + rng.normal(0, 8, page.shape), 0, 255).astype(np.uint8)
    ramp = np.linspace(1.0, 0.24, 640, dtype=np.float32)[None, :]                 # свет 1 → 0.24 слева направо
    vign = 1 - 0.25 * ((np.arange(480) - 240.0) / 240.0) ** 2
    lamp = np.clip(page * ramp * vign[:, None] + rng.normal(0, 6, page.shape), 0, 255).astype(np.uint8)
    logo_page = page.copy(); cv2.rectangle(logo_page, (430, 300), (600, 450), 0, -1)   # логотип 170 × 150
    gt_logo = np.where(logo_page < 128, 0, 255).astype(np.uint8)
    lamp_logo = np.clip(logo_page * ramp * vign[:, None] + rng.normal(0, 6, page.shape), 0, 255).astype(np.uint8)
    return gt, flat, lamp, gt_logo, lamp_logo


gt_text, flat, page, gt_logo, page_logo = make_pages()
accuracy = lambda binary, gt: 100.0 * float((binary == gt).mean())

t_otsu, otsu = cv2.threshold(page, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
four = {"порог 128": cv2.threshold(page, 128, 255, cv2.THRESH_BINARY)[1],
        "Otsu, t = %d" % t_otsu: otsu,
        "adaptive mean 31, C 15": cv2.adaptiveThreshold(page, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 31, 15),
        "adaptive gauss 31, C 15": cv2.adaptiveThreshold(page, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)}
for name, b in four.items():
    print("%-24s верных пикселей %5.1f %%" % (name, accuracy(b, gt_text)))
print("для сравнения: Otsu на странице с ровным светом — t = %d, %.1f %%" % (
    cv2.threshold(flat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0], accuracy(cv2.threshold(flat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1], gt_text)))
viz.grid(dict([("страница под лампой", page)] + list(four.items())[1:]), cols=4, size=3.0)

Адаптивный порог выигрывает с разгромом, но у него два параметра: `blockSize` (окно локального
среднего, нечётное) и `C` (запас против шума на пустой бумаге). Числа 31 и 15 на слайде взялись не из
воздуха — их выбрали **по метрике**. Повторите этот выбор.

Напишите `sweep_adaptive(page, gt, block_sizes, Cs, method=cv2.ADAPTIVE_THRESH_MEAN_C)`: для каждой
пары (`blockSize`, `C`) — `cv2.adaptiveThreshold(page, 255, method, cv2.THRESH_BINARY, blockSize, C)`
и доля верных пикселей против `gt` в процентах. Результат — массив `float` формы
`(len(block_sizes), len(Cs))`: строки — `blockSize`, столбцы — `C`.

In [ ]:
def sweep_adaptive(page, gt, block_sizes, Cs, method=cv2.ADAPTIVE_THRESH_MEAN_C):
    """Доля верных пикселей (%) адаптивного порога для каждой пары (blockSize, C). → массив (len(block_sizes), len(Cs))."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 6–8 строк: пустой массив, два цикла, adaptiveThreshold, сравнение с gt через mean
    raise NotImplementedError("TODO 3")
    # ───────────────────────────────────────────────────────────


BLOCK_SIZES, CS = (5, 11, 31, 91, 301), (0, 5, 10, 15, 20, 30, 50)
table = sweep_adaptive(page, gt_text, BLOCK_SIZES, CS)
assert table.shape == (len(BLOCK_SIZES), len(CS)), "ожидался массив (blockSize × C)"
assert np.all((table >= 0) & (table <= 100)), "доля верных пикселей — в процентах, 0…100"
assert abs(table[BLOCK_SIZES.index(31), CS.index(15)] - accuracy(four["adaptive mean 31, C 15"], gt_text)) < 1e-6, "клетка (31, 15) должна совпасть с таблицей выше"
assert table.max() > 99, "лучшая пара должна давать > 99 %"
assert table[:, CS.index(0)].max() < 85, "при C = 0 шум на пустой бумаге должен ронять точность ниже 85 %"

i_best, j_best = np.unravel_index(np.argmax(table), table.shape)
print("TODO 3 ✔  · лучшая пара: blockSize %d, C %d → %.1f %%" % (BLOCK_SIZES[i_best], CS[j_best], table.max()))
print("\n%-10s" % "bs \\ C" + "".join("%7d" % C for C in CS))
for bs, row in zip(BLOCK_SIZES, table):
    print("%-10d" % bs + "".join("%7.1f" % v for v in row))

### Переносится ли подобранное?

Урок занятия: **параметр, подобранный на одной картинке, надо проверять на другой.** Берём лучшую пару
и применяем к странице с логотипом — тем же светом, тем же шрифтом, плюс чёрный прямоугольник 170 × 150.
Заодно проверяем два «народных средства»: «выровняй свет CLAHE — и Otsu справится» и деление на фон
(размытую копию страницы).

In [ ]:
bs_best, C_best = BLOCK_SIZES[i_best], CS[j_best]
logo = (slice(300, 451), slice(430, 601))                    # где стоит логотип
inside_black = lambda b: 100.0 * float((b[logo] == 0).mean())

adaptive = lambda img, bs, C: cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, bs, C)
otsu_of = lambda img: cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
divide_bg = lambda img: cv2.divide(img, cv2.medianBlur(img, 51), scale=255)            # страница ÷ размытый фон

print("%-34s %14s %16s" % ("страница с логотипом", "верных", "логотип чёрный"))
variants = {"adaptive %d, C %d (лучшая)" % (bs_best, C_best): adaptive(page_logo, bs_best, C_best),
            "adaptive 301, C 15": adaptive(page_logo, 301, 15),
            "Otsu": otsu_of(page_logo),
            "CLAHE 2.0 → Otsu": otsu_of(cv2.createCLAHE(2.0, (8, 8)).apply(page_logo)),
            "÷ фон (medianBlur 51) → Otsu": otsu_of(divide_bg(page_logo))}
for name, b in variants.items():
    print("%-34s %12.1f %% %14.0f %%" % (name, accuracy(b, gt_logo), inside_black(b)))
print("\nта же проверка на странице без логотипа: CLAHE → Otsu %.1f %%, ÷ фон → Otsu %.1f %%"
      % (accuracy(otsu_of(cv2.createCLAHE(2.0, (8, 8)).apply(page)), gt_text), accuracy(otsu_of(divide_bg(page)), gt_text)))
viz.grid({"страница с логотипом": page_logo, **{k: v for k, v in list(variants.items())[:2]}, "÷ фон → Otsu": variants["÷ фон (medianBlur 51) → Otsu"]}, cols=4, size=3.0)

### Вопрос, на который отвечаем вслух

Лучшая пара с чистой страницы на странице с логотипом выедает логотип почти целиком (столбец «логотип
чёрный») — почему, и какой из двух параметров за это отвечает? `blockSize = 301` логотип спасает,
но текст читает хуже.
CLAHE «выравнивает свет», но Otsu после него всё равно проигрывает адаптивному вдвое — что CLAHE
делает с гистограммой, а чего не делает? И почему деление на фон работает лучше всех — какое допущение
о странице оно использует?

---

## ⭐ Звёздочки *(по желанию, +0.5, до конца следующей недели)*

**⭐ GrabCut на своём фото.** Слайд 46: прямоугольник вокруг объекта → две смеси гауссиан «объект/фон»
+ разрез графа. Ячейка ниже пускает его на кропе «лампы» вокруг найденной рамки и сравнивает с маской
`inRange` по IoU. Звёздочка — повторить на **своём фото** предмета на пёстром фоне: прямоугольник задать
руками, сравнить GrabCut с `color_mask` по маске, нарисованной глазами (или по числу «дыр» и компонент),
и написать три строки: где какой метод выигрывает.

In [ ]:
(x, y, w, h), _ = largest_component(m_lamp)
pad = 40
crop_lamp, crop_gt = lamp[y - pad:y + h + pad, x - pad:x + w + pad], gt_ball[y - pad:y + h + pad, x - pad:x + w + pad]
gc = np.zeros(crop_lamp.shape[:2], np.uint8)
bgd, fgd = np.zeros((1, 65), np.float64), np.zeros((1, 65), np.float64)
t0 = time.perf_counter()
cv2.grabCut(crop_lamp, gc, (pad - 20, pad - 20, w + 40, h + 40), bgd, fgd, 5, cv2.GC_INIT_WITH_RECT)
dt = 1000 * (time.perf_counter() - t0)
fg = (gc == cv2.GC_FGD) | (gc == cv2.GC_PR_FGD)
iou_mask = lambda m, g: float((m & g).sum()) / float((m | g).sum())
print("GrabCut на кропе %s: %.0f мс, IoU с эталоном %.3f · маска inRange + морфология: IoU %.3f"
      % (crop_lamp.shape[:2], dt, iou_mask(fg, crop_gt), iou_mask(m_lamp[y - pad:y + h + pad, x - pad:x + w + pad] > 0, crop_gt)))
viz.grid({"кроп «лампы»": crop_lamp, "GrabCut": fg.astype(np.uint8) * 255, "inRange + морфология": m_lamp[y - pad:y + h + pad, x - pad:x + w + pad]}, cols=3, size=3.0)

**⭐ Подбор HSV-диапазона по клику.** Границы «с эталона» мы брали по известной маске; в жизни
эталон показывают мышью. Ячейка ниже (только локально, `RUN_PICKER = True`) открывает окно с камерой:
клик — добавить квадрат 15 × 15 вокруг точки в эталон, `r` — сбросить, `q` — выйти; в консоль печатаются
границы `hsv_box_from` по всем кликам. Звёздочка — прогнать с ними `color_mask` + `largest_component`
на своей камере и сдать ноутбук с картинкой, границами и долей кадров, где объект найден.

In [ ]:
RUN_PICKER = False                                          # True — открыть окно (не в Colab)
if RUN_PICKER and not cvcourse.IN_COLAB:
    picked, cur = [], {}
    def on_click(event, px, py, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            picked.append(cur["frame"][max(py - 7, 0):py + 8, max(px - 7, 0):px + 8].reshape(-1, 3))
            print("эталон: %d пикселей · границы HSV:" % sum(len(p) for p in picked), *hsv_box_from(np.vstack(picked)))
    cv2.namedWindow("picker"); cv2.setMouseCallback("picker", on_click)
    for frame in cio.video_frames(0, resize=(640, 360)):
        cur["frame"] = frame
        view = frame if not picked else cv2.addWeighted(frame, 0.6, cv2.cvtColor(color_mask(frame, *hsv_box_from(np.vstack(picked))), cv2.COLOR_GRAY2BGR), 0.4, 0)
        cv2.imshow("picker", view)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("r"): picked.clear()
        if key == ord("q"): break
    cv2.destroyAllWindows()

---

## Мост к домашнему заданию

**ДЗ 1 «Фотолаборатория», выдача на этой неделе (W3), дедлайн W6, клиника — занятие 5.** Сегодняшние
функции входят в него почти без изменений:

| Сегодня | В ДЗ 1 |
|---------|--------|
| `color_mask` + `largest_component` | пункт 2, **хромакей**: маска объекта в HSV → морфология → крупнейшая компонента → подстановка фона |
| границы H и S с эталона, V открыт | обоснование выбора цветового пространства в отчёте — с числами recall/precision, как сегодня |
| гистограммы под маской (`calcHist`) | пункт 1: гистограммы **до и после** гаммы, баланса белого, CLAHE |
| «CLAHE → Otsu» и «÷ фон» | отчёт: почему CLAHE лучше `equalizeHist` на *вашем* фото — ответ измерением, не цитатой |
| таблица «параметр → метрика» | тот же приём для гаммы и clipLimit: не «стало красиво», а число |

Чего сегодня **не** делали и придётся сделать дома: обход папки снимков, гамма и баланс белого «серый
мир» (слайд 17 L3), геометрия (после L4 и занятия 4), отчёт. Полное ТЗ — `homeworks/hw1-photolab/README.md`.

## Зачёт за занятие

Ассистент ставит зачёт, если:

- [ ] ячейка проверки окружения прошла;
- [ ] **TODO 1–3 выполнены**, все ассерты проходят;
- [ ] вы можете объяснить, почему границы V не берут с эталона и что делает открытие, а что закрытие;
- [ ] на вопрос про логотип (какой параметр адаптивного порога его выедает и почему) есть внятный ответ.

**2 % итоговой оценки**, ставится в классе. Досылать ничего не нужно.